In [1]:
RUN_TARGET = "local"  # "colab" | "local"


## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1.
2. Run the install cell and restart if requested.
3. Mount Drive if you want to read synced `models/` and `results/` artifacts.
4. Run the remaining cells top to bottom.

### Running locally
1. Keep `RUN_TARGET = "local"`.
2. The notebook scans the local `models/` and `results/` folders.
3. It loads saved probe artifacts and renders main-paper and supplementary figures without retraining or reprobe computation.


In [2]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "scipy": "1.15.3",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "statsmodels": "0.14.5",
        "seaborn": "0.13.2",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Local environment detected. Skipping Colab setup.


In [3]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


Local run: skipping Google Drive mount.


# 08 - Cross-Model Probe Visualisation

This notebook is the centralized visualization layer for probe outputs that already exist in `results/`. It does not load models or generate attribution scores; it only discovers saved probe-row CSVs, combines them, and renders main-paper plus supplementary figures.


In [4]:
import sys
from pathlib import Path

# Allow imports from the src-layout package without a root-level shim.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break
import importlib

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    _SRC_DIR = RUNTIME_ROOT / "src"
    if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils

importlib.reload(baseline_notebook_utils)
importlib.reload(mtl_epitope_notebook_utils)

from xallergen.baseline_notebook_utils import configure_matplotlib_cache, detect_device, find_project_root, print_runtime_context
from xallergen.mtl_epitope_notebook_utils import discover_probe_row_artifacts, load_available_probe_rows, render_probe_figures_from_rows, save_combined_probe_tables

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd


/Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


RUN_TARGET: local
Device: mps
Project root: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0
GPU configuration:
  backend: Apple Metal Performance Shaders (MPS)
  built with MPS: True
  MPS available: True


## Discover Available Variants

Each compatible checkpoint in `models/` is mapped to the probe-row CSV that should already exist in `results/`. If a probe artifact is missing, this notebook skips that model rather than recomputing it. Use `07_generate_probe_rows.ipynb` to generate or refresh missing probe artifacts first.


In [6]:
discovery_df = discover_probe_row_artifacts(MODELS_DIR, RESULTS_DIR)
discovery_df


,checkpoint_name,variant,model_family,checkpoint_path,probe_rows_path,probe_rows_exists
0,baseline_frozen_esm2.pt,baseline,Baseline (04),/Users/jianzhouyao/Library/Mobile Documents/co...,/Users/jianzhouyao/Library/Mobile Documents/co...,True
1,mtl_frozen_esm2_epitope.pt,frozen,MTL (05 frozen),/Users/jianzhouyao/Library/Mobile Documents/co...,/Users/jianzhouyao/Library/Mobile Documents/co...,True
2,mtl_top1_unfrozen_esm2_epitope.pt,top1_unfrozen,MTL (06 top1_unfrozen),/Users/jianzhouyao/Library/Mobile Documents/co...,/Users/jianzhouyao/Library/Mobile Documents/co...,True


In [7]:
all_probe_df = load_available_probe_rows(discovery_df)
all_probe_df.head()


,accession,seq_len,epitope_density,n_epitope_residues,model_family,method,label_variant,auroc,auprc,precision_at_k,source_probe_rows_path
0,A0A834K244,251,0.549801,138,Baseline (04),attention_weights,original,0.630242,0.621814,0.623188,/Users/jianzhouyao/Library/Mobile Documents/co...
1,A0A834K244,251,0.549801,138,Baseline (04),attention_weights,scrambled,0.491920,0.547467,0.543478,/Users/jianzhouyao/Library/Mobile Documents/co...
2,A0A834K244,251,0.549801,138,Baseline (04),integrated_gradients,original,0.474349,0.535056,0.492754,/Users/jianzhouyao/Library/Mobile Documents/co...
3,A0A834K244,251,0.549801,138,Baseline (04),integrated_gradients,scrambled,0.505130,0.569623,0.550725,/Users/jianzhouyao/Library/Mobile Documents/co...
4,A0A834K244,251,0.549801,138,Baseline (04),gradient_x_input,original,0.488906,0.539134,0.543478,/Users/jianzhouyao/Library/Mobile Documents/co...


## Save Combined Tables

These artifacts provide a single reusable row-wise table plus bootstrap-CI summaries across all available families. Figures can be regenerated later from `all_models_probing_rows.csv` without rerunning training or probing.

In [8]:
table_outputs = save_combined_probe_tables(all_probe_df, RESULTS_DIR)
MULTI_MODEL_ROWS_CSV = table_outputs["rows_path"]
MULTI_MODEL_SUMMARY_CSV = table_outputs["summary_path"]
summary_df = table_outputs["summary_df"]
summary_df


Saved main localization summary to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/probing/summaries/main_localization_summary.csv
Saved supplementary all-signals summary to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/probing/summaries/supplementary_all_signals_summary.csv
Saved label-scrambling summary to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/probing/summaries/probe_summary_with_scrambling.csv


,model_family,method,label_variant,method_category,auroc_mean,auroc_ci_low,auroc_ci_high,auprc_mean,auprc_ci_low,auprc_ci_high,precision_at_k_mean,precision_at_k_ci_low,precision_at_k_ci_high,n_proteins
0,Baseline (04),random_mean,original,Null baseline,0.5011,0.4993,0.5029,0.2697,0.2222,0.3226,0.2517,0.2041,0.3044,58
1,Baseline (04),random_mean,scrambled,Null baseline,0.5004,0.4990,0.5019,0.2690,0.2220,0.3219,0.2515,0.2039,0.3044,58
2,Baseline (04),attention_weights,original,Model-internal signal,0.4438,0.4151,0.4752,0.2468,0.1985,0.3008,0.2131,0.1622,0.2703,58
3,Baseline (04),attention_weights,scrambled,Model-internal signal,0.5172,0.5011,0.5332,0.2778,0.2317,0.3267,0.2618,0.2153,0.3130,58
4,Baseline (04),integrated_gradients,original,Post-hoc attribution,0.4328,0.3932,0.4772,0.2550,0.2046,0.3103,0.2189,0.1633,0.2776,58
5,Baseline (04),integrated_gradients,scrambled,Post-hoc attribution,0.4974,0.4821,0.5115,0.2653,0.2195,0.3161,0.2510,0.2013,0.3056,58
6,Baseline (04),gradient_x_input,original,Post-hoc attribution,0.4627,0.4411,0.4862,0.2572,0.2106,0.3096,0.2465,0.1964,0.3023,58
7,Baseline (04),gradient_x_input,scrambled,Post-hoc attribution,0.5023,0.4850,0.5195,0.2750,0.2261,0.3280,0.2482,0.1990,0.3016,58
8,Baseline (04),smoothgrad_ig,original,Post-hoc attribution,0.4330,0.3929,0.4771,0.2548,0.2045,0.3102,0.2198,0.1636,0.2788,58
9,Baseline (04),smoothgrad_ig,scrambled,Post-hoc attribution,0.5036,0.4848,0.5215,0.2670,0.2182,0.3221,0.2523,0.2029,0.3085,58


## Plot All Available Families

In [9]:
figure_outputs = render_probe_figures_from_rows(all_probe_df, RESULTS_DIR)
figure_outputs


Saved plot to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main/main_localization_auroc.png
Saved plot to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main/main_localization_auroc.pdf
Saved plot to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main/main_localization_auprc.png
Saved plot to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main/main_localization_auprc.pdf
Saved plot to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main/main_localization_precision_at_k.png
Saved plot to: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/

{'main_dir': PosixPath('/Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main'),
 'supplementary_dir': PosixPath('/Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/supplementary'),
 'main_base': PosixPath('/Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main/main_localization.png'),
 'supplementary_base': PosixPath('/Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/supplementary/supplementary_all_signals.png'),
 'scrambling_base': PosixPath('/Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/supplementary/label_scrambling_sanity_check.png')}

In [10]:
print("Saved combined artifacts:")
for out_path in [MULTI_MODEL_ROWS_CSV, MULTI_MODEL_SUMMARY_CSV]:
    print(f"  {out_path}")
for label, out_path in figure_outputs.items():
    print(f"  {label}: {out_path}")


Saved combined artifacts:
  /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/probing/rows/all_models_probing_rows.csv
  /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/probing/summaries/all_models_probing_summary.csv
  main_dir: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main
  supplementary_dir: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/supplementary
  main_base: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/main/main_localization.png
  supplementary_base: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/results/figures/supplementary/supplementary_all_signal